# 实践 17.1  模拟序列的变异效应评分与饱和突变扫描

《统计遗传与智能育种学》第十七章 · 电子资源 `17_regulatory_variant.ipynb`

本 notebook 的代码与书中“完整Python代码”逐字一致。运行检查：重启内核并运行全部单元（Restart Kernel and Run All）。

电子资源：17_regulatory_variant.ipynb。Notebook构造带已知motif的正/负序列，先划分训练/测试，再只用训练序列拟合k-mer词表与逻辑回归分类器，避免测试集参与特征学习。随后对一条从未参与训练的新序列逐位做三种单碱基替换，输出预测概率差，并检查高效应位点是否回到已知motif区域。本例用逻辑回归演示序列分类及概率差评分。

运行检查：重启内核并运行全部单元  
本书验证环境实际结果见下方代码输出；必须同时报告独立测试受试者工作特征曲线下面积（area under the receiver operating characteristic curve, ROC AUC）、平均精确率（AP）、饱和突变高分替换记录与真实motif区间重合情况。

## 输入、完整代码、实际输出与解释

输入与可复现条件：SEED=1701；600条正序列+600条负序列；100 bp模拟序列；7 bp已知基序；70%/30%训练—测试；无外部数据文件。

In [1]:
import sys, platform
import numpy as np
import pandas as pd
import sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import train_test_split

SEED = 1701
rng = np.random.default_rng(SEED)
print('Python', sys.version.split()[0])
print('Platform', platform.platform())
print('numpy', np.__version__, 'pandas', pd.__version__, 'scikit-learn', sklearn.__version__)
print('SEED', SEED)

DNA = np.array(list('ACGT'))
MOTIF = 'ACGTGCA'
L = 100

def random_seq():
    return ''.join(rng.choice(DNA, L))

pos, neg = [], []
for _ in range(600):
    s = list(random_seq())
    start = int(rng.integers(12, L-len(MOTIF)-12))
    s[start:start+len(MOTIF)] = list(MOTIF)
    pos.append(''.join(s))
    neg.append(random_seq())
seq = np.array(pos + neg)
y = np.r_[np.ones(len(pos), dtype=int), np.zeros(len(neg), dtype=int)]
idx = np.arange(len(y))
tr, te = train_test_split(idx, test_size=0.30, stratify=y, random_state=SEED)

# 关键：k-mer词表只在训练集拟合，测试序列不参与任何特征学习。
vec = CountVectorizer(analyzer='char', ngram_range=(5,7), lowercase=False, min_df=2)
Xtr = vec.fit_transform(seq[tr])
Xte = vec.transform(seq[te])
clf = LogisticRegression(max_iter=1500, C=1.0).fit(Xtr, y[tr])
prob = clf.predict_proba(Xte)[:,1]
print('train/test', len(tr), len(te), 'vocabulary', len(vec.vocabulary_))
print('ROC_AUC', round(roc_auc_score(y[te], prob), 3),
      'average_precision', round(average_precision_score(y[te], prob), 3))

# 对一条全新的、未参与训练的序列做单碱基饱和突变。
ref_list = list(random_seq())
motif_start = 45
ref_list[motif_start:motif_start+len(MOTIF)] = list(MOTIF)
ref = ''.join(ref_list)
pref = clf.predict_proba(vec.transform([ref]))[0,1]
rows = []
for i, ref_base in enumerate(ref):
    for alt in DNA:
        if alt == ref_base:
            continue
        alt_seq = ref[:i] + alt + ref[i+1:]
        palt = clf.predict_proba(vec.transform([alt_seq]))[0,1]
        rows.append((i, ref_base, alt, palt-pref, motif_start <= i < motif_start+len(MOTIF)))
sat = pd.DataFrame(rows, columns=['pos','ref','alt','delta_prob','inside_true_motif'])
top = sat.iloc[np.argsort(-sat.delta_prob.abs().to_numpy())[:15]]
print('reference_probability', round(float(pref), 4), 'true_motif_interval', f'{motif_start}-{motif_start+len(MOTIF)-1}')
print(top.round(4).to_string(index=False))
print('top15_inside_true_motif', int(top.inside_true_motif.sum()), '/ 15')


Python 3.12.13
Platform macOS-26.5.2-arm64-arm-64bit
numpy 2.3.5 pandas 2.2.3 scikit-learn 1.8.0
SEED 1701
train/test 840 360 vocabulary 20558
ROC_AUC 1.0 average_precision 1.0
reference_probability 0.9786 true_motif_interval 45-51
 pos ref alt  delta_prob  inside_true_motif
  48   T   A     -0.9763               True
  47   G   C     -0.9760               True
  48   T   C     -0.9760               True
  47   G   T     -0.9758               True
  48   T   G     -0.9757               True
  49   G   T     -0.9756               True
  47   G   A     -0.9753               True
  49   G   A     -0.9750               True
  49   G   C     -0.9746               True
  46   C   G     -0.9707               True
  46   C   T     -0.9703               True
  50   C   G     -0.9690               True
  46   C   A     -0.9678               True
  50   C   T     -0.9638               True
  50   C   A     -0.9629               True
top15_inside_true_motif 15 / 15


结果解释：模型只在训练集学习k-mer词表，因此测试ROC AUC和AP来自完整留出的测试集。饱和突变能否把高效应替换记录集中到预埋motif，是一个受控单元测试；即使命中，也只说明模型学习到了预设序列规则，不能据此证明真实作物性状的因果效应。

可修改练习：把MOTIF长度改为5或9，或把正样本数降到150；比较ROC AUC、average precision和前15条替换记录位于真实基序内的比例。